# Clase 3 · 04 · Ventanas por tiempo del evento
Grano de Gold: una fila por ventana de 5 minutos y canal de pago, con ventanas ya finalizadas.
Deduplicación y agregación se separan en dos consultas Delta para observar y recuperar cada estado.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala de clases 1 y 2")
dbutils.widgets.dropdown("expected_batch_id", "stream_001",
                         ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "03 - Etapa a validar")
dbutils.widgets.text("job_run_id", "interactive", "04 - ID de ejecución")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_support

In [ ]:
from pyspark.sql import functions as F
config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
repetitions = STREAM_REPETITIONS[config.scale]
expected_batch_id = normalize_stream_batch(dbutils.widgets.get("expected_batch_id"))
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
spark.conf.set("spark.sql.session.timeZone", "UTC")
require_stream_lab(spark, config)

## 1. Agregar con watermark
`window(event_ts, '5 minutes')` crea intervalos [inicio, fin): un evento exactamente a 12:05
pertenece a [12:05, 12:10). Se agrupa por tiempo del evento, no por el momento de ingesta.
El control de 005 debe llegar al watermark; por eso no se filtra antes de esta agregación.
Sus métricas de negocio son cero y su fila agregada se filtra al finalizar.

In [ ]:
events = spark.readStream.table(qualified_table(config, "silver_stream_events"))
purchase = F.col("event_type") == "purchase"
windows = (events.withWatermark("event_ts", "10 minutes")
           .groupBy(F.window("event_ts", "5 minutes"), "payment_channel")
           .agg(F.sum(F.when(purchase, 1).otherwise(0)).alias("event_count"),
                F.sum(F.when(purchase, F.col("amount")).otherwise(F.lit(0)))
                 .cast("decimal(18,2)").alias("total_amount"),
                F.sum(F.when(purchase, F.col("is_fraud")).otherwise(0)).alias("fraud_count"))
           .where(F.col("event_count") > 0)
           .select(F.col("window.start").alias("window_start"),
                   F.col("window.end").alias("window_end"), "payment_channel",
                   "event_count", "total_amount", "fraud_count"))
assert windows.isStreaming

## 2. Emitir sólo ventanas cerradas
Usamos append: Gold no muestra las ventanas abiertas todavía almacenadas en estado.
Después de 001 y 002, Gold puede estar vacía y eso es esperado.
003 cierra las primeras ventanas; 005 permite cerrar [12:25, 12:30).
Esperar sin eventos nuevos no avanza por sí mismo el watermark.

In [ ]:
query = (windows.writeStream.format("delta").outputMode("append")
         .option("checkpointLocation", stream_checkpoint(config, "gold"))
         .queryName(f"{config.schema}_gold")
         .trigger(availableNow=True)
         .toTable(qualified_table(config, "gold_stream_windows")))
wait_and_record_stream(spark, config, query, "gold", job_run_id)

In [ ]:
%sql
SELECT *, CASE WHEN event_count > 0 THEN fraud_count * 1.0 / event_count END AS fraud_rate
FROM gold_stream_windows ORDER BY window_start, payment_channel

### Para responder
¿Por qué no es correcto reconciliar Gold con todos los eventos de Silver antes de cerrar ventanas?
¿Qué pasaría si descartaras el control antes de withWatermark? ¿Y si usaras complete en lugar de append?